In [ ]:
"""
Modèle de recommandation (VENTES) — Content-Based Filtering
============================================================
"""

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

DATA_PATH = "dataset_ventes_final.csv"

DISPLAY_COLUMNS = ["titre", "ville", "quartier", "prix_dh", "surface_m2", "chambres"]


def detect_features(df: pd.DataFrame) -> tuple[list, list]:
    """
    Détecte automatiquement les colonnes features depuis le CSV.
    Retourne (features_numériques, features_catégorielles_one_hot).
    """
    exclude = {
        "prix_dh", "source", "titre", "ville", "quartier",
        "listing_id", "url", "description", "equipements",
    }
    all_features = [c for c in df.columns if c not in exclude]
    cat_features = [c for c in all_features if c.startswith("ville_")]
    num_features = [c for c in all_features if c not in cat_features]
    return num_features, cat_features


class PropertyRecommenderVentes:
    """Recommandeur content-based pour les annonces de VENTE."""

    def __init__(self):
        self.scaler = StandardScaler()
        self.catalog: pd.DataFrame | None = None
        self.feature_matrix: np.ndarray | None = None
        self.similarity_matrix: np.ndarray | None = None
        self.numeric_features: list = []
        self.categorical_features: list = []

    def fit(self, df: pd.DataFrame) -> "PropertyRecommenderVentes":
        self.catalog = df.reset_index(drop=True)
        self.numeric_features, self.categorical_features = detect_features(self.catalog)

        print("Features détectées :")
        print("  Numériques ({}): {}".format(len(self.numeric_features), self.numeric_features))
        print("  Catégorielles ({}): {}".format(len(self.categorical_features), self.categorical_features))

        numeric = self.catalog[self.numeric_features].astype(float)
        numeric_scaled = self.scaler.fit_transform(numeric)

        if self.categorical_features:
            categorical = self.catalog[self.categorical_features].astype(int).to_numpy()
            self.feature_matrix = np.hstack([numeric_scaled, categorical])
        else:
            self.feature_matrix = numeric_scaled

        self.similarity_matrix = cosine_similarity(self.feature_matrix)
        return self

    def recommend_similar_to(self, index: int, top_n: int = 5) -> pd.DataFrame:
        """Biens les plus similaires à celui d'index donné (hors lui-même)."""
        scores = self.similarity_matrix[index]
        order = np.argsort(scores)[::-1]
        order = [i for i in order if i != index][:top_n]
        result = self.catalog.iloc[order][DISPLAY_COLUMNS].copy()
        result["score_similarite"] = scores[order]
        return result

    def recommend_from_preferences(self, preferences: dict, top_n: int = 5) -> pd.DataFrame:
        """Recommandation à partir de critères explicites."""
        # Construction d'un dict avec TOUTES les features numériques
        # Les flags *_imputed sont mis à 0 (valeurs réelles fournies par l'utilisateur)
        query_values = {}

        # Valeurs par défaut = médianes du catalogue
        for col in self.numeric_features:
            if col == "prix_dh":
                query_values[col] = preferences.get("budget_dh", self.catalog["prix_dh"].median())
            elif col == "surface_m2":
                query_values[col] = preferences.get("surface_m2", self.catalog["surface_m2"].median())
            elif col == "chambres":
                query_values[col] = preferences.get("chambres", self.catalog["chambres"].median())
            elif col == "salles_de_bain":
                query_values[col] = preferences.get("salles_de_bain", self.catalog["salles_de_bain"].median())
            elif col == "quartier_prix_estime":
                query_values[col] = self._quartier_prix_estime_for_ville(preferences.get("ville"))
            elif col.endswith("_imputed"):
                # Pour une requête utilisateur, on considère que la valeur est réelle
                query_values[col] = 0
            else:
                # Fallback : médiane
                query_values[col] = self.catalog[col].median()

        # Construire le DataFrame dans le bon ordre
        numeric_query = pd.DataFrame([query_values[col] for col in self.numeric_features]).T
        numeric_query.columns = self.numeric_features
        numeric_query_scaled = self.scaler.transform(numeric_query)

        # One-hot pour la ville demandée
        if self.categorical_features:
            ville_query = np.zeros((1, len(self.categorical_features)))
            ville_col_name = "ville_{}".format(preferences.get("ville", "Autre"))
            if ville_col_name in self.categorical_features:
                ville_query[0, self.categorical_features.index(ville_col_name)] = 1
            else:
                autre_col = "ville_Autre"
                if autre_col in self.categorical_features:
                    ville_query[0, self.categorical_features.index(autre_col)] = 1

            query_vector = np.hstack([numeric_query_scaled, ville_query])
        else:
            query_vector = numeric_query_scaled

        scores = cosine_similarity(query_vector, self.feature_matrix)[0]
        order = np.argsort(scores)[::-1][:top_n]

        result = self.catalog.iloc[order][DISPLAY_COLUMNS].copy()
        result["score_similarite"] = scores[order]
        return result

    def _quartier_prix_estime_for_ville(self, ville: str | None) -> float:
        if ville is None:
            return self.catalog["quartier_prix_estime"].median()
        subset = self.catalog[self.catalog["ville"] == ville]
        if len(subset) == 0:
            return self.catalog["quartier_prix_estime"].median()
        return subset["quartier_prix_estime"].median()


def main():
    df = pd.read_csv(DATA_PATH)

    recommender = PropertyRecommenderVentes()
    recommender.fit(df)

    print("\nCatalogue chargé : {} biens".format(len(df)))

    # --- Démo 1 : biens similaires ---
    demo_index = df.sample(1, random_state=7).index[0]
    reference = df.iloc[demo_index][DISPLAY_COLUMNS]
    print("\n=== DEMO 1 — Biens similaires à une annonce consultée ===")
    print("Annonce de référence :")
    print(reference.to_string())

    similar = recommender.recommend_similar_to(demo_index, top_n=5)
    print("\nBiens similaires recommandés :")
    print(similar.to_string(index=False))

    # --- Démo 2 : recommandation par préférences ---
    print("\n=== DEMO 2 — Recommandation à partir de critères ===")
    preferences = {
        "ville": "Casablanca",
        "budget_dh": 2500000,
        "surface_m2": 120,
        "chambres": 3,
        "salles_de_bain": 2,
    }
    print("Critères : {}".format(preferences))

    recs = recommender.recommend_from_preferences(preferences, top_n=5)
    print("\nBiens recommandés :")
    print(recs.to_string(index=False))

    # --- Sauvegarde ---
    joblib.dump(recommender, "recommender_ventes_artifacts.joblib")
    print("\nRecommandeur sauvegardé : recommender_ventes_artifacts.joblib")

    # --- Rapport ---
    report_lines = [
        "# Rapport — modèle de recommandation VENTES (content-based, v1)",
        "",
        "## Approche",
        "Content-based filtering : chaque bien à vendre est représenté par un vecteur de",
        "caractéristiques standardisées, comparées par similarité cosinus.",
        "",
        "## Catalogue",
        "- {} biens (fusion Mubawab + Sarouty Ventes)".format(len(df)),
        "- Features numériques ({}): {}".format(len(recommender.numeric_features), ", ".join(recommender.numeric_features)),
        "- Features catégorielles ({}): {}".format(len(recommender.categorical_features), ", ".join(recommender.categorical_features)),
        "",
        "## Exemple — Démo 1 (biens similaires)",
        "Référence :",
        df.iloc[demo_index][DISPLAY_COLUMNS].to_string(),
        "",
        "Recommandations :",
        similar.to_string(index=False),
        "",
        "## Exemple — Démo 2 (à partir de préférences)",
        "Critères : {}".format(preferences),
        "",
        "Recommandations :",
        recs.to_string(index=False),
        "",
        "## Limites",
        "- Purement content-based (pas de collaborative filtering)",
        "- Pas d'extraction NLP sur la description",
        "- Le prix étant une feature de similarité, deux biens similaires mais avec des",
        "  prix très différents auront un score faible — souhaitable pour respecter le budget",
        "",
        "## Prochaine étape",
        "Exposer via `/predict/recommandation-ventes` dans le micro-service IA.",
        "",
        "---",
        "*Généré automatiquement*",
    ]

    with open("rapport_recommandation_ventes.md", "w", encoding="utf-8") as f:
        f.write("\n".join(report_lines))

    print("Rapport sauvegardé : rapport_recommandation_ventes.md")


if __name__ == "__main__":
    main()

Features détectées :
  Numériques (7): ['chambres', 'chambres_imputed', 'quartier_prix_estime', 'salles_de_bain', 'salles_de_bain_imputed', 'surface_m2', 'surface_m2_imputed']
  Catégorielles (21): ['ville_Agadir', 'ville_Autre', 'ville_Bouskoura', 'ville_Casablanca', 'ville_Dar Bouazza', 'ville_El Jadida', 'ville_El Mansouria', 'ville_Fès', 'ville_Inconnu', 'ville_Kénitra', 'ville_Marrakech', 'ville_Martil', 'ville_Mohammedia', 'ville_Médiouna', 'ville_Nouaceur', 'ville_Rabat', 'ville_Salé', 'ville_Skhirat', 'ville_Tanger', 'ville_Temara', 'ville_Zenata']

Catalogue chargé : 2470 biens

=== DEMO 1 — Biens similaires à une annonce consultée ===
Annonce de référence :
titre         Vente Appartement - Dar Bouazza
ville                              Casablanca
quartier                          Dar Bouazza
prix_dh                             1900000.0
surface_m2                               80.0
chambres                                  3.0

Biens similaires recommandés :
                